In [273]:
options(scipen=999)
pacman::p_load(plyr,dplyr, stringr, data.table, tidyr,
               pheatmap, colorRamps, gridExtra, ggplot2, ggrepel, RColorBrewer,vioplot,ggforce,
              gridExtra, ggplot2, ggrepel, RColorBrewer,vioplot,grid, patchwork, grid, ggh4x, ggalluvial,
               ggsankey)


## Directories

In [274]:
# Set working directories
base.dir =  "/nfs/lab/projects/CellCrosstalk/npod.pancreas/Final/"
assets.dir = paste(base.dir, "Assets/", sep = "") 
results.dir = paste(base.dir, "Results/corrected/", sep = "")
permutation.result.dir = paste(base.dir, "Results/corrected.permutation.test/", sep = "")


# Databases
cc.db = "/nfs/lab/projects/islet_multiomics_stress/analysis/CellChat/Assets/CellChatDB.human.txt"
Gaulton.db.dir = "/nfs/lab/Luca/Assets/LR.Database/GaultonDB_V2.txt"

permutations.test.tmp.dir = paste(permutation.result.dir, "TMP/", sep = "")
permutations.test.res.dir = paste(permutation.result.dir, "Results/", sep = "")

cc.db = "/nfs/lab/projects/islet_multiomics_stress/analysis/CellChat/Assets/CellChatDB.human.txt"

## Variables

In [275]:
# conditions
condition.ls = c("ND", "Aab", "T1D_early","T1D_late")

colors.conditions = c("ND" = "#85EF1B",
                      "Aab" = "#EFEF1B",
                      "T1D_early" = "#EF851B",
                      "T1D_late" = "#EF1B1B")

# Contrasts
Contrast_1 = c(condition.ls[2], condition.ls[3], condition.ls[4])
Contrast_2 = c(condition.ls[1], condition.ls[1], condition.ls[1])

# Celltypes and compartments
Endocrine = c('Alpha', 'Beta', 'Delta')
Exocrine = c('Acinar1_2_6','Acinar_3','Acinar_4','Acinar_5','Ductal','MUC5b_Ductal')
Immune = c('Tcells','Macrophage')
Endothelial = c('Endothelial')
Stromal = c('Activated_Stellate','Quiescent_Stellate')

cell.pop.order = c(Endocrine, Exocrine, Immune, Endothelial, Stromal)
rev.cell.pop.order = c(rev(Stromal), rev(Endothelial), rev(Immune), rev(Exocrine), rev(Endocrine), use.names = TRUE)

compartment = cell.pop.order
compartment = ifelse(compartment  %in%  Endocrine, "Endocrine", 
                 ifelse(compartment  %in%  Exocrine, "Exocrine",
                        ifelse(compartment  %in%  Immune, "Immune",
                               ifelse(compartment  %in%  Endothelial, "Endothelial", "Stromal"))))

Cellpop.compartment = as.data.frame(cbind(cell.pop.order, compartment))
Cellpop.compartment$compartment = factor(Cellpop.compartment$compartment, levels = c('Endocrine','Exocrine','Immune','Endothelial', "Stromal"))
colnames(Cellpop.compartment)[1] = "Celltype"

colors.compartments = c('Endocrine'='#26bfbf',
           'Exocrine'='#ed872d',
           'Immune'='#3F98E0',
           'Endothelial'='#800080',
           'Stromal'='#F5DE6C')

Cellpop.compartment$colors.compartments = compartment
Cellpop.compartment$colors.compartments = mapvalues(Cellpop.compartment$colors.compartments, names(colors.compartments), colors.compartments, warn_missing = TRUE)
gaps.compartments = c(3, 9, 11, 12)

In [276]:
# Load and prepare Gaulton DB
Gaulton.db.dir = "/nfs/lab/Luca/Assets/LR.Database/GaultonDB_V3.txt"
# Load list of ligands and receptors
SM.ls.df = read.table(paste(assets.dir, "SM.ls.df.txt", sep=""), sep = "\t", stringsAsFactors = F, header = T)
SM.ls = unique(SM.ls.df$gene)
length(SM.ls)

# Load database
cc.db.data = read.table(cc.db, sep = "\t", stringsAsFactors = F, header = T)
# cc.db.data = cc.db.data[,c(11,2,3)]
message("CCDB number of interactions: ", length(levels(factor(cc.db.data$interaction_name))))

gaulton.db.data = read.table(Gaulton.db.dir, sep = "\t", stringsAsFactors = F, header = T)
colnames(gaulton.db.data)[1] = "gene"

SM.ls.df.GDB = merge(SM.ls.df, gaulton.db.data, by = "gene", all.x = TRUE)
SM.ls.df.GDB[is.na(SM.ls.df.GDB)] <- ""

# Calculate category sizes
categories.size = SM.ls.df.GDB[SM.ls.df.GDB$Type == "Ligand",]

# Ligands
ligands.GDB = filter(SM.ls.df.GDB, Type != "Receptor")
colnames(ligands.GDB)[1] = "ligand"
head(ligands.GDB)


Functional.classification.order = c('Hormones & Neuropeptides', 'Enzymes', 'Developmental Proteins', 
                                    'Growth Factors and Cytokines', 'Antigen-Presenting Molecules', 'Complement System',
                                    'Immune regulators', 'Modulators', 'Cell Adhesion Molecule', 'ECM Proteins')

[1] 857

CCDB number of interactions: 1939



,ligand,Type,Mode.of.Action,Signaling,Functional.classification,Functional.sub.classification,Gene.Family
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,ADCYAP1,Ligand,Secreted Signaling,Hormones & Neuropeptides,Other,Other,PACAP Family
2,ADGRE5,Ligand,Cell-Cell Contact,CAM,Other,Other Cell Adhesion Proteins,-
3,ADIPOQ,Ligand,Secreted Signaling,Growth Factors & Cytokines,Other Cytokines,Other Cytokines,Adipokine Family
4,ADM,Ligand,Secreted Signaling,Hormones & Neuropeptides,Hormone-like (Heart),Hormone (Heart),Adrenomedullin Family
5,AGRN,Ligand,ECM-Receptor,ECM,Other,Other ECM proteins,Agrin Family
6,AGT,Ligand,Secreted Signaling,Hormones & Neuropeptides,Hormone (Liver),Hormone (Liver),Angiotensinogen Family


# Permutation test

## Observations

In [277]:
LR.data.consensus.sig.thresh = read.table(paste(results.dir,
                                               "LR.data.consensus.sig.thresh.txt", sep = ""),
                                          sep = "\t", stringsAsFactors = F, header = T)
LR.data.consensus.sig.thresh = LR.data.consensus.sig.thresh[LR.data.consensus.sig.thresh$target %in% cell.pop.order, ]
LR.data.consensus.sig.thresh = LR.data.consensus.sig.thresh[LR.data.consensus.sig.thresh$source %in% cell.pop.order, ]

In [278]:
# Split LR pairs
LR.data.ND = LR.data.consensus.sig.thresh[LR.data.consensus.sig.thresh$condition == condition.ls[1], ]
LR.data.AAB = LR.data.consensus.sig.thresh[LR.data.consensus.sig.thresh$condition == condition.ls[2], ]
LR.data.ETD = LR.data.consensus.sig.thresh[LR.data.consensus.sig.thresh$condition == condition.ls[3], ]
LR.data.LTD = LR.data.consensus.sig.thresh[LR.data.consensus.sig.thresh$condition == condition.ls[4], ]
nrow(LR.data.consensus.sig.thresh)
# Print stats
message("LR pairs in ND: ", nrow(LR.data.ND))
message("LR pairs in AAB : ", nrow(LR.data.AAB))
message("LR pairs in Early T1D: ", nrow(LR.data.ETD))
message("LR pairs in Late T1D: ", nrow(LR.data.LTD))

[1] 87650

LR pairs in ND: 27139

LR pairs in AAB : 27459

LR pairs in Early T1D: 18179

LR pairs in Late T1D: 14873



In [279]:
head(LR.data.consensus.sig.thresh, n=1)

,ligand,source,target,receptor,prob,pval,interaction_name,interaction_name_2,pathway_name,annotation,⋯,FDR,interaction.id,compartment.source,compartment.target,Type,Mode.of.Action,Signaling,Functional.classification,Functional.sub.classification,Gene.Family
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,⋯,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,ADCYAP1,Beta,Quiescent_Stellate,ADCYAP1R1,0.00006164112,0.0001,ADCYAP1_ADCYAP1R1,ADCYAP1 - ADCYAP1R1,PACAP,Secreted Signaling,⋯,0.000153715,Beta--Quiescent_Stellate--ADCYAP1--ADCYAP1R1,Endocrine,Stromal,Ligand,Secreted Signaling,Hormones & Neuropeptides,Other,Other,PACAP Family


In [280]:
condition.sizes = c(nrow(LR.data.ND), nrow(LR.data.AAB), nrow(LR.data.ETD), nrow(LR.data.LTD))
condition.sizes

condition.size_1 = c(condition.sizes[2], condition.sizes[3], condition.sizes[4])
condition.size_2 = c(condition.sizes[1], condition.sizes[1], condition.sizes[1])

[1] 27139 27459 18179 14873

In [281]:
nsim = 100

In [282]:
# Create a named vector that maps conditions to sizes
condition_mapping <- c("ND" = condition.sizes[1], "Aab" = condition.sizes[2],
                       "T1D_early" = condition.sizes[3], "T1D_late" = condition.sizes[4])

In [283]:
LR.data.consensus.use = LR.data.consensus.sig.thresh
nrow(LR.data.consensus.use)

[1] 87650

In [284]:
head(LR.data.consensus.use, n = 2)

,ligand,source,target,receptor,prob,pval,interaction_name,interaction_name_2,pathway_name,annotation,⋯,FDR,interaction.id,compartment.source,compartment.target,Type,Mode.of.Action,Signaling,Functional.classification,Functional.sub.classification,Gene.Family
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,⋯,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
1,ADCYAP1,Beta,Quiescent_Stellate,ADCYAP1R1,0.00006164112,0.0001,ADCYAP1_ADCYAP1R1,ADCYAP1 - ADCYAP1R1,PACAP,Secreted Signaling,⋯,0.000153715,Beta--Quiescent_Stellate--ADCYAP1--ADCYAP1R1,Endocrine,Stromal,Ligand,Secreted Signaling,Hormones & Neuropeptides,Other,Other,PACAP Family
2,ADCYAP1,Beta,Activated_Stellate,ADCYAP1R1,0.00022605242,0.0001,ADCYAP1_ADCYAP1R1,ADCYAP1 - ADCYAP1R1,PACAP,Secreted Signaling,⋯,0.000153715,Beta--Activated_Stellate--ADCYAP1--ADCYAP1R1,Endocrine,Stromal,Ligand,Secreted Signaling,Hormones & Neuropeptides,Other,Other,PACAP Family


In [285]:
# Perform the aggregation
aggregated_data <- aggregate(prob ~ Signaling + source + target + condition + compartment.source + compartment.target, LR.data.consensus.sig.thresh, sum)
# normalize by condition size
aggregated_data <- aggregated_data %>%
  dplyr::mutate(prob.norm = prob / condition_mapping[as.character(condition)]) %>%
  dplyr::mutate(condition.size = condition_mapping[as.character(condition)])
aggregated_data$compartment.source = factor(aggregated_data$compartment.source, levels = unique(compartment))
aggregated_data$compartment.target = factor(aggregated_data$compartment.target, levels = unique(compartment))
aggregated_data$classification.source.target = paste(aggregated_data$Signaling, aggregated_data$source, aggregated_data$target, sep = "--")

In [286]:
head(filter(aggregated_data, condition == "ND"))

,Signaling,source,target,condition,compartment.source,compartment.target,prob,prob.norm,condition.size,classification.source.target
,<chr>,<chr>,<chr>,<chr>,<fct>,<fct>,<dbl>,<dbl>,<int>,<chr>
1,CAM,Alpha,Alpha,ND,Endocrine,Endocrine,0.1547314336,0.000005701441969,27139,CAM--Alpha--Alpha
2,Developmental,Alpha,Alpha,ND,Endocrine,Endocrine,0.0119908891,0.000000441832385,27139,Developmental--Alpha--Alpha
3,ECM,Alpha,Alpha,ND,Endocrine,Endocrine,0.0166483944,0.000000613449075,27139,ECM--Alpha--Alpha
4,Enzymatic,Alpha,Alpha,ND,Endocrine,Endocrine,0.0001232913,0.000000004542958,27139,Enzymatic--Alpha--Alpha
5,Growth Factors & Cytokines,Alpha,Alpha,ND,Endocrine,Endocrine,0.0018826797,0.000000069371740,27139,Growth Factors & Cytokines--Alpha--Alpha
6,Hormones & Neuropeptides,Alpha,Alpha,ND,Endocrine,Endocrine,0.0369990901,0.000001363318107,27139,Hormones & Neuropeptides--Alpha--Alpha


In [287]:
diff.obs = list()
abs.diff.obs = list()
obs.interaction.consensus = list()

for (i in seq_along(Contrast_1)){
    LR.data.use <- aggregated_data
    # Subset for the contrast
    contrast = c(Contrast_1[i], Contrast_2[i])
    # Create consensus
    interaction.consensus = filter(aggregated_data, condition %in% contrast) %>%
        select(classification.source.target) %>%
        dplyr::distinct()
    obs.interaction.consensus[[i]] = interaction.consensus
    message("Processing ", Contrast_1[i], " vs ", Contrast_2[i])
    LR.data.c1 = LR.data.use[LR.data.use$condition %in% contrast[1], ]
    LR.data.c2 = LR.data.use[LR.data.use$condition %in% contrast[2], ]
    message("Distribution of interactions in contrast: \n",
            Contrast_1[i], ": ", nrow(LR.data.c1), "\n",
            Contrast_2[i], ": ", nrow(LR.data.c2))
    # Complete dataset
    LR.data.c1 = merge(LR.data.c1, interaction.consensus, by = "classification.source.target", all.y = TRUE)
    LR.data.c2 = merge(LR.data.c2, interaction.consensus, by = "classification.source.target", all.y = TRUE)
    # Replace NA with 0
    LR.data.c1$prob.norm[is.na(LR.data.c1$prob.norm)] = 0
    LR.data.c2$prob.norm[is.na(LR.data.c2$prob.norm)] = 0 
    message("Distribution of interactions in contrast - Number should be the same - : \n",
            Contrast_1[i], ": ", nrow(LR.data.c1), "\n",
            Contrast_2[i], ": ", nrow(LR.data.c2))
    # Order 
    LR.data.c1 = arrange(LR.data.c1, classification.source.target)
    LR.data.c2 = arrange(LR.data.c2, classification.source.target)
    message("Are Dataframe ordered in the same way? ", identical(LR.data.c1$classification.source.target, LR.data.c2$classification.source.target))
    # Compute difference
    diff = LR.data.c1$prob.norm - LR.data.c2$prob.norm
    diff.obs[[i]] = data.frame(classification.source.target = LR.data.c1$classification.source.target,
                               condition = contrast[1],
                               diff = diff, abs.diff = abs(diff))
}

Processing Aab vs ND

Distribution of interactions in contrast: 
Aab: 1255
ND: 1231

Distribution of interactions in contrast - Number should be the same - : 
Aab: 1286
ND: 1286

Are Dataframe ordered in the same way? TRUE

Processing T1D_early vs ND

Distribution of interactions in contrast: 
T1D_early: 1016
ND: 1231

Distribution of interactions in contrast - Number should be the same - : 
T1D_early: 1267
ND: 1267

Are Dataframe ordered in the same way? TRUE

Processing T1D_late vs ND

Distribution of interactions in contrast: 
T1D_late: 849
ND: 1231

Distribution of interactions in contrast - Number should be the same - : 
T1D_late: 1257
ND: 1257

Are Dataframe ordered in the same way? TRUE



## Simulations

In [288]:
setwd(permutations.test.tmp.dir)
sim.list <- dir(pattern = "_DFnet.txt")

In [289]:
# Create tables to fill up
diff.sim = list()
random.success.table = list()
# Create tables with correct number of rows
random.success.table[[1]] = data.frame(classification.source.target = obs.interaction.consensus[[1]])
random.success.table[[2]] = data.frame(classification.source.target = obs.interaction.consensus[[2]])
random.success.table[[3]] = data.frame(classification.source.target = obs.interaction.consensus[[3]])

for (s.i in seq_along(sim.list)){
        # Get files
        print(paste0("Processing: ", sim.list[s.i]))
        # Load simulation data
        LR.data.use <- read.table(sim.list[s.i], sep = "\t", stringsAsFactors = F, header = T)
        # add functional classification info
        LR.data.use = merge(LR.data.use, ligands.GDB, by = "ligand")
        message("\t Detected: ", nrow(LR.data.use), " interactions")
        # Create a named vector that maps conditions to sizes
        condition.sim_mapping <- c("ND" = nrow(LR.data.use[LR.data.use$condition == "ND",]),
                               "Aab" = nrow(LR.data.use[LR.data.use$condition == "Aab",]),          
                               "T1D_early" = nrow(LR.data.use[LR.data.use$condition == "T1D_early",]),           
                               "T1D_late" = nrow(LR.data.use[LR.data.use$condition == "T1D_late",]))
        message(" Normalizing ND: ", condition.sim_mapping[1],
                "\n Normalizing AAB: ", condition.sim_mapping[2],
                "\n Normalizing ETD: ", condition.sim_mapping[3],
                "\n Normalizing LTD: ", condition.sim_mapping[4]
               )        
        # Perform the aggregation
        aggregated_data.sim <- aggregate(prob ~ Signaling + source + target + condition, LR.data.use, sum)
        # normalize by condition size
        aggregated_data.sim <- aggregated_data.sim %>%
          dplyr::mutate(prob.norm = prob / condition.sim_mapping[as.character(condition)]) %>%
          dplyr::mutate(condition.size = condition_mapping[as.character(condition)])
        aggregated_data.sim$classification.source.target = paste(aggregated_data.sim$Signaling, aggregated_data.sim$source, aggregated_data.sim$target, sep = "--")
        for (i in seq_along(Contrast_1)){
                        # Subset for the contrast
                        contrast = c(Contrast_1[i], Contrast_2[i])
                        message("Processing ", Contrast_1[i], " vs ", Contrast_2[i])
                        # Use consensus of the observations to compare the same category
                        interaction.consensus = obs.interaction.consensus[[i]]
                        LR.data.c1 = aggregated_data.sim[aggregated_data.sim$condition %in% contrast[1], ]
                        LR.data.c2 = aggregated_data.sim[aggregated_data.sim$condition %in% contrast[2], ]
                        message("Distribution of interactions in contrast: \n",
                                Contrast_1[i], ": ", nrow(LR.data.c1), "\n",
                                Contrast_2[i], ": ", nrow(LR.data.c2))
                        # Complete dataset
                        LR.data.c1 = merge(LR.data.c1, interaction.consensus, by = "classification.source.target", all.y = TRUE)
                        LR.data.c2 = merge(LR.data.c2, interaction.consensus, by = "classification.source.target", all.y = TRUE)
                        # Replace NA with 0
                        LR.data.c1$prob.norm[is.na(LR.data.c1$prob.norm)] = 0
                        LR.data.c2$prob.norm[is.na(LR.data.c2$prob.norm)] = 0 
                        message("Distribution of interactions in contrast - Number should be the same - : \n",
                                Contrast_1[i], ": ", nrow(LR.data.c1), "\n",
                                Contrast_2[i], ": ", nrow(LR.data.c2))
                        # Order 
                        LR.data.c1 = arrange(LR.data.c1, classification.source.target)
                        LR.data.c2 = arrange(LR.data.c2, classification.source.target)
                        message("Are Dataframe ordered in the same way? ", identical(LR.data.c1$classification.source.target, LR.data.c2$classification.source.target))
                        # Compute difference
                        diff = LR.data.c1$prob.norm - LR.data.c2$prob.norm
                        diff.sim[[i]] = data.frame(classification.source.target = LR.data.c1$classification.source.target, condition = contrast[1],
                                        diff = diff, abs.diff = abs(diff))
                        random.success = diff.sim[[i]]$abs.diff > diff.obs[[i]]$abs.diff
                        random.success.table[[i]] = cbind(random.success.table[[i]], random.success)
                        colnames(random.success.table[[i]])[ncol(random.success.table[[i]])] = s.i  
                }
}

[1] "Processing: CellChat_Permutation_n1_DFnet.txt"


	 Detected: 99223 interactions

 Normalizing ND: 21962
 Normalizing AAB: 27346
 Normalizing ETD: 26497
 Normalizing LTD: 23418

Processing Aab vs ND

Distribution of interactions in contrast: 
Aab: 1238
ND: 1176

Distribution of interactions in contrast - Number should be the same - : 
Aab: 1286
ND: 1286

Are Dataframe ordered in the same way? TRUE

Processing T1D_early vs ND

Distribution of interactions in contrast: 
T1D_early: 1241
ND: 1176

Distribution of interactions in contrast - Number should be the same - : 
T1D_early: 1267
ND: 1267

Are Dataframe ordered in the same way? TRUE

Processing T1D_late vs ND

Distribution of interactions in contrast: 
T1D_late: 1217
ND: 1176

Distribution of interactions in contrast - Number should be the same - : 
T1D_late: 1257
ND: 1257

Are Dataframe ordered in the same way? TRUE



[1] "Processing: CellChat_Permutation_n10_DFnet.txt"


	 Detected: 100143 interactions

 Normalizing ND: 27234
 Normalizing AAB: 25830
 Normalizing ETD: 23698
 Normalizing LTD: 23381

Processing Aab vs ND

Distribution of interactions in contrast: 
Aab: 1227
ND: 1248

Distribution of interactions in contrast - Number should be the same - : 
Aab: 1286
ND: 1286

Are Dataframe ordered in the same way? TRUE

Processing T1D_early vs ND

Distribution of interactions in contrast: 
T1D_early: 1210
ND: 1248

Distribution of interactions in contrast - Number should be the same - : 
T1D_early: 1267
ND: 1267

Are Dataframe ordered in the same way? TRUE

Processing T1D_late vs ND

Distribution of interactions in contrast: 
T1D_late: 1212
ND: 1248

Distribution of interactions in contrast - Number should be the same - : 
T1D_late: 1257
ND: 1257

Are Dataframe ordered in the same way? TRUE



[1] "Processing: CellChat_Permutation_n100_DFnet.txt"


...

	 Detected: 100686 interactions

 Normalizing ND: 25551
 Normalizing AAB: 26512
 Normalizing ETD: 25835
 Normalizing LTD: 22788

Processing Aab vs ND

Distribution of interactions in contrast: 
Aab: 1233
ND: 1227

Distribution of interactions in contrast - Number should be the same - : 
Aab: 1286
ND: 1286

Are Dataframe ordered in the same way? TRUE

Processing T1D_early vs ND

Distribution of interactions in contrast: 
T1D_early: 1240
ND: 1227

Distribution of interactions in contrast - Number should be the same - : 
T1D_early: 1267
ND: 1267

Are Dataframe ordered in the same way? TRUE

Processing T1D_late vs ND

Distribution of interactions in contrast: 
T1D_late: 1223
ND: 1227

Distribution of interactions in contrast - Number should be the same - : 
T1D_late: 1257
ND: 1257

Are Dataframe ordered in the same way? TRUE



[1] "Processing: CellChat_Permutation_n98_DFnet.txt"


	 Detected: 98776 interactions

 Normalizing ND: 27324
 Normalizing AAB: 24853
 Normalizing ETD: 21963
 Normalizing LTD: 24636

Processing Aab vs ND

Distribution of interactions in contrast: 
Aab: 1216
ND: 1241

Distribution of interactions in contrast - Number should be the same - : 
Aab: 1286
ND: 1286

Are Dataframe ordered in the same way? TRUE

Processing T1D_early vs ND

Distribution of interactions in contrast: 
T1D_early: 1164
ND: 1241

Distribution of interactions in contrast - Number should be the same - : 
T1D_early: 1267
ND: 1267

Are Dataframe ordered in the same way? TRUE

Processing T1D_late vs ND

Distribution of interactions in contrast: 
T1D_late: 1228
ND: 1241

Distribution of interactions in contrast - Number should be the same - : 
T1D_late: 1257
ND: 1257

Are Dataframe ordered in the same way? TRUE



[1] "Processing: CellChat_Permutation_n99_DFnet.txt"


	 Detected: 100458 interactions

 Normalizing ND: 28236
 Normalizing AAB: 24233
 Normalizing ETD: 24409
 Normalizing LTD: 23580

Processing Aab vs ND

Distribution of interactions in contrast: 
Aab: 1218
ND: 1259

Distribution of interactions in contrast - Number should be the same - : 
Aab: 1286
ND: 1286

Are Dataframe ordered in the same way? TRUE

Processing T1D_early vs ND

Distribution of interactions in contrast: 
T1D_early: 1207
ND: 1259

Distribution of interactions in contrast - Number should be the same - : 
T1D_early: 1267
ND: 1267

Are Dataframe ordered in the same way? TRUE

Processing T1D_late vs ND

Distribution of interactions in contrast: 
T1D_late: 1217
ND: 1259

Distribution of interactions in contrast - Number should be the same - : 
T1D_late: 1257
ND: 1257

Are Dataframe ordered in the same way? TRUE



In [295]:
# Wrap up test results
for (i in seq_along(diff.obs)){
    message("Processing contrast #", i)
    diff.obs[[i]]$random.success = rowSums(random.success.table[[i]] == "TRUE", na.rm = TRUE)
    diff.obs[[i]]$pvalue = diff.obs[[i]]$random.success/nsim
    # correct pvalues of what had no difference to begin with
    diff.obs[[i]]$pvalue = replace(diff.obs[[i]]$pvalue, diff.obs[[i]]$diff == 0, 1)
    # calculate FDR
    diff.obs[[i]]$pvalue[diff.obs[[i]]$pvalue == 0] = 0.0001
    diff.obs[[i]]$FDR = p.adjust(diff.obs[[i]]$pvalue, 'BH')
    # counting stars
    diff.obs[[i]]$stars = diff.obs[[i]]$FDR
    diff.obs[[i]]$stars[diff.obs[[i]]$FDR > 0.1]<-""
    diff.obs[[i]]$stars[diff.obs[[i]]$FDR < 0.1]<-"*"
    diff.obs[[i]]$stars[diff.obs[[i]]$FDR < 0.05]<-"**"
    diff.obs[[i]]$stars[diff.obs[[i]]$FDR < 0.01]<-"***"
    file.name = paste(permutations.test.res.dir, "Permutation.test_cellpairs_signaling.", Contrast_1[i], ".res", sep = "")
    write.table(diff.obs[[i]], file.name, sep = "\t", quote = F, col.names=T, row.names = T) 
}

Processing contrast #1

Processing contrast #2

Processing contrast #3



In [296]:
test = filter(diff.obs[[2]], pvalue < 0.05)
nrow(test)
test = filter(diff.obs[[2]], FDR < 0.1)
nrow(test)

[1] 299

[1] 182

In [312]:
cell.pairs.classification = NULL
setwd(permutations.test.res.dir)

for (i in dir(pattern = "Permutation.test_cellpairs_signaling.")){
    message(i)
    data.use = read.table(i, sep = "\t")
    cell.pairs.classification = rbind(cell.pairs.classification, data.use)
}

Permutation.test_cellpairs_signaling.Aab.res

Permutation.test_cellpairs_signaling.T1D_early.res

Permutation.test_cellpairs_signaling.T1D_late.res



In [313]:
# Add metadata
cell.pairs.classification$classification = str_split_fixed(cell.pairs.classification$classification.source.target, pattern = "--", n = 3)[,1]
cell.pairs.classification$source = str_split_fixed(cell.pairs.classification$classification.source.target, pattern = "--", n = 3)[,2]
cell.pairs.classification$target = str_split_fixed(cell.pairs.classification$classification.source.target, pattern = "--", n = 3)[,3]
cell.pairs.classification = merge(cell.pairs.classification, Cellpop.compartment, 
                                  by.x = "source", by.y = "Celltype")
cell.pairs.classification$compartment = factor(cell.pairs.classification$compartment, levels = unique(compartment))
cell.pairs.classification$source = factor(cell.pairs.classification$source, levels = cell.pop.order)
cell.pairs.classification$target = factor(cell.pairs.classification$target, levels = cell.pop.order)

# Linear transformation
# normalize applying linear transformation
cell.pairs.classification$diff = cell.pairs.classification$diff * 10000
cell.pairs.classification$abs.diff = cell.pairs.classification$abs.diff * 10000

head(cell.pairs.classification, n= 2)

,source,classification.source.target,condition,diff,abs.diff,random.success,pvalue,FDR,stars,classification,target,compartment,colors.compartments
,<fct>,<chr>,<chr>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<chr>,<chr>,<fct>,<fct>,<chr>
1,Acinar_3,Enzymatic--Acinar_3--Endothelial,T1D_early,0.015080276135,0.015080276135,11,0.11,0.3203908,,Enzymatic,Endothelial,Exocrine,#ed872d
2,Acinar_3,Enzymatic--Acinar_3--Tcells,Aab,0.000007828976,0.000007828976,100,1.00,1.0000000,NA,Enzymatic,Tcells,Exocrine,#ed872d


In [314]:
file.name = paste(permutations.test.res.dir, "Permutation.test_ALL_cellpairs_signaling.res", sep = "")
write.table(cell.pairs.classification, file.name, sep = "\t", quote = F, col.names=T, row.names = T)

# Plot

In [318]:
# Normalize adding the same 10k to the differece
# add labels to the sankey plot
# Show sankey plots for both beta to all and all to beta

In [320]:
cell.use = "Beta"
condition.ls

[1] "ND"        "Aab"       "T1D_early" "T1D_late"

In [348]:
plot.data = cell.pairs.classification %>%
        dplyr::mutate(source.classification = paste(source, classification, sep = "--"))%>%
        dplyr::mutate(direction = ifelse(diff > 0, "up", "down"))%>%
        dplyr::filter(source == "Beta" &
                      condition %in% condition.ls[3] &
                      !classification %in% c("Other"))

In [349]:
plot.data$diff.scaled <- scale(plot.data$diff)

In [351]:
ggplot(plot.data, aes(axis1 = classification, axis2 = target, y = diff)) +
  geom_alluvium(aes(fill = classification, alpha = -log(FDR)), color = "black") +
  geom_stratum() +
  geom_text(stat = "stratum", aes(label = after_stat(stratum))) +
  facet_grid(direction ~ source, scales = "free", space = "free") +
  theme_bw()

ERROR while rich displaying an object: Error in `combine_vars()`:
! At least one layer must contain all faceting variables: `up or down`
✖ Plot is missing `up or down`
✖ Layer 1 is missing `up or down`
✖ Layer 2 is missing `up or down`
✖ Layer 3 is missing `up or down`
✖ Layer 4 is missing `up or down`

Traceback:
1. tryCatch(withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler), error = outer_handler)
2. tryCatchList(expr, classes, parentenv, handlers)
3. tryCatchOne(expr, names, parentenv, handlers[[1L]])
4. doTryCatch(return(expr), name, parentenv, handler)
5. withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](o

In [85]:
# Perform the aggregation
aggregated_data <- aggregate(prob ~ Functional.classification + source + target + condition + compartment.source + compartment.target, LR.data.consensus.sig.thresh, sum)
# normalize by condition size
aggregated_data <- aggregated_data %>%
  dplyr::mutate(prob.norm = prob / condition_mapping[as.character(condition)]) %>%
  dplyr::mutate(condition.size = condition_mapping[as.character(condition)])
aggregated_data$compartment.source = factor(aggregated_data$compartment.source, levels = unique(compartment))
aggregated_data$compartment.target = factor(aggregated_data$compartment.target, levels = unique(compartment))
aggregated_data$classification.source.target = paste(aggregated_data$Functional.classification, aggregated_data$source, aggregated_data$target, sep = "--")

In [ ]:
# Plot datasets as is
# Plot differential signaling?!

In [86]:
aggregated_data

Functional.classification,source,target,condition,compartment.source,compartment.target,prob,prob.norm,condition.size,classification.source.target
<chr>,<chr>,<chr>,<chr>,<fct>,<fct>,<dbl>,<dbl>,<int>,<chr>
Cell Adhesion Molecule,Alpha,Alpha,Aab,Endocrine,Endocrine,0.16098081135,0.000005862588272,27459,Cell Adhesion Molecule--Alpha--Alpha
Developmental Proteins,Alpha,Alpha,Aab,Endocrine,Endocrine,0.01121850205,0.000000408554647,27459,Developmental Proteins--Alpha--Alpha
ECM Proteins,Alpha,Alpha,Aab,Endocrine,Endocrine,0.01798352315,0.000000654922727,27459,ECM Proteins--Alpha--Alpha
Enzymes,Alpha,Alpha,Aab,Endocrine,Endocrine,0.00003752931,0.000000001366740,27459,Enzymes--Alpha--Alpha
Growth Factors and Cytokines,Alpha,Alpha,Aab,Endocrine,Endocrine,0.00232782120,0.000000084774435,27459,Growth Factors and Cytokines--Alpha--Alpha
Hormones & Neuropeptides,Alpha,Alpha,Aab,Endocrine,Endocrine,0.03032926571,0.000001104529142,27459,Hormones & Neuropeptides--Alpha--Alpha
Modulators,Alpha,Alpha,Aab,Endocrine,Endocrine,0.00003645492,0.000000001327613,27459,Modulators--Alpha--Alpha
Cell Adhesion Molecule,Beta,Alpha,Aab,Endocrine,Endocrine,0.14967617611,0.000005450896832,27459,Cell Adhesion Molecule--Beta--Alpha
Developmental Proteins,Beta,Alpha,Aab,Endocrine,Endocrine,0.01446340781,0.000000526727405,27459,Developmental Proteins--Beta--Alpha


In [ ]:
# flow chart sample data
plot.data = LR.data.consensus.sig.thresh %>% 
    dplyr::filter(condition == condition.ls[3]) %>%
    dplyr::mutate(interaction.group = paste(source, Functional.classification, target, sep = "--")) %>%
    dplyr::group_by(interaction.group) %>%
    dplyr::summarize(prob.norm = sum(prob, na.rm = TRUE) / first(condition.size)) %>%
    dplyr::mutate(source = str_split_fixed(interaction.group, "--", 3)[,1],
                            classification = str_split_fixed(interaction.group, "--", 3)[,2],
                            target = str_split_fixed(interaction.group, "--", 3)[,3],
                            logFDR = 1) %>% 
              dplyr::ungroup()